# Modular Answer-Script OCR & Grading Pipeline (Rubric v2, Kaggle-Ready)

This notebook implements a complete 6-stage vision-language pipeline for transcribing and automatically grading handwritten exam scripts for **NCTB HSC English 1st Paper (Class 11)**.
It features:
- **Multi-Page Script Segmentation & Question Routing**: Automatically identifies and groups pages from multi-page exam PDFs (15–20+ pages) into discrete questions (Q3 Summary, Q7 Paragraph, Q8 Graph/Chart, Q9 Story, Q10 Letter/Email, Q11 Theme) using `extraction.csv` manifests or automated VLM detection.
- **6-Stage Modular Grading Engine**: Independent, toggleable stages for Cold OCR, Rubric Alignment, Reference-Primed OCR, Visual Adjudication, Chief Examiner Scoring (`rubric_v2.txt`), and Compressor Auditing.
- **Structural Constraints & Hard Caps**: Enforces mandatory ceilings for off-topic drift, length violations, verbatim copying, and missing layout elements.
- **Traceable Evaluation Output**: Exports structured results matching `evaluation.csv` format.

> **Important:** Ensure **Internet is Enabled** in Kaggle Notebook Settings to install dependencies and download HuggingFace model checkpoints.

In [ ]:
# Step 1: Install Pinned Dependencies
!pip install --quiet --upgrade \
    "transformers>=4.48.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.0" \
    "pypdfium2>=4.30.0" \
    "pymupdf>=1.24.0" \
    "pillow>=10.4.0" \
    "pydantic>=2.8.0" \
    "pyyaml>=6.0.1" \
    "peft>=0.13.0" \
    "trl>=0.12.0" \
    "scikit-learn>=1.5.0" \
    "scipy>=1.13.0"

# Optional: Install vLLM if on supported Linux GPU tier
# !pip install --quiet "vllm>=0.6.0"

In [ ]:
# Step 2: Environment Check & Path Setup
import os
import sys
import torch
import yaml
import json
from PIL import Image

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Configure paths
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
OUTPUT_DIR = os.path.join(WORKING_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory initialized: {OUTPUT_DIR}")

In [ ]:
# Step 3: Define Pipeline Configuration (YAML)
config_yaml = f"""
pipeline:
  enabled_stages:
    extractor_a: true
    rubric_aligner: true
    extractor_b: true
    ocr_supervisor: true
    examiner: true
    compressor: true

model:
  backend: mock            # vllm | transformers | mock
  checkpoint: google/gemma-3-27b-it
  quantization: w4a16     # w4a16 | bitsandbytes_4bit | none
  kv_cache: fp8
  temperature: 0.0
  max_tokens: 4096

ingestion:
  pdf_dpi: 300
  max_image_side: 2048
  page_router: true

language:
  supported: [bn, en]

rubric_path: rubric_v2.txt

logging:
  output_dir: {OUTPUT_DIR}
  save_per_stage_json: true
"""

config_path = os.path.join(WORKING_DIR, "config.yaml")
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_yaml)
print(f"Configuration written to {config_path}")

In [ ]:
# Step 4: Import Pipeline Engine & Schemas
sys.path.insert(0, os.path.join(os.getcwd(), "code", "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from schemas import PipelineConfig, EnabledStagesConfig, TASK_MAX_MARKS
from orchestrator import run_pipeline, run_script_pipeline, export_evaluation_csv
from ingestion.pdf_loader import load_images_from_file
from ingestion.page_router import PageRouter, load_manifest_from_csv
from eval.ocr_metrics import compute_cer, compute_wer
from eval.grading_metrics import compute_all_grading_metrics, benchmark_against_ground_truth

config = PipelineConfig.from_dict(yaml.safe_load(config_yaml))
print("Engine successfully initialized.")

In [ ]:
# Step 5: Multi-Page Script Routing & Grading on a Full PDF Submission
sample_pdf = "datasets/SE_11_Q1_0001.pdf"
manifest_csv = "extraction.csv"
output_csv = os.path.join(OUTPUT_DIR, "evaluation_results.csv")

if os.path.exists(sample_pdf):
    print(f"Processing full exam script: {sample_pdf}")
    results = run_script_pipeline(
        pdf_path=sample_pdf,
        config=config,
        manifest_csv=manifest_csv,
        auto_route=True,
        verbose=True
    )
    export_evaluation_csv(results, output_csv)
    print(f"\nGraded {len(results)} questions. CSV report exported to: {output_csv}")
else:
    print("Sample PDF not found, running mock single question execution...")
    dummy_pages = [Image.new("RGB", (1200, 1600), color=(255, 255, 255))]
    res = run_pipeline(
        pages=dummy_pages,
        rubric="Rubric v2 Specification",
        reference_solution="Look at the chart: USA electricity in 1980.",
        config=config,
        task_type="Graph_Chart",
        max_mark=10.0,
        task_id="CHART_001",
        student_id="SE_11_Q1_0001",
        verbose=True
    )
    results = [res]
    export_evaluation_csv(results, output_csv)

In [ ]:
# Step 6: Stage Ablation Benchmark
print("\n" + "="*90)
print("STAGE ABLATION MATRIX BENCHMARK (RUBRIC V2)")
print("="*90)

dummy_pages = [Image.new("RGB", (1000, 1400), color=(255, 255, 255))]
ablation_configs = [
    ("Full Pipeline (All 6 Stages ON)", {}),
    ("Ablation: Stage 2 (RubricAligner) OFF", {"rubric_aligner": False}),
    ("Ablation: Stage 3 (Extractor B) OFF", {"extractor_b": False}),
    ("Ablation: Stage 4 (OCR Supervisor) OFF", {"ocr_supervisor": False}),
    ("Ablation: Stages 3 & 4 both OFF (Cheap OCR)", {"extractor_b": False, "ocr_supervisor": False}),
    ("Ablation: Stage 6 (Compressor) OFF", {"compressor": False}),
    ("Ablation: Minimal (Stage 1 + Stage 5 only)", {
        "rubric_aligner": False, "extractor_b": False, "ocr_supervisor": False, "compressor": False
    })
]

print(f"{'Configuration':<45} | {'Final Score':<12} | {'Band':<10} | {'Cap Reason':<18}")
print("-" * 90)

for desc, overrides in ablation_configs:
    cfg_dict = config.to_dict()
    for k, v in overrides.items():
        cfg_dict["pipeline"]["enabled_stages"][k] = v
    
    abl_cfg = PipelineConfig.from_dict(cfg_dict)
    abl_cfg.logging.save_per_stage_json = False
    
    r = run_pipeline(
        pages=dummy_pages,
        rubric="Rubric v2 Specification",
        reference_solution="Look at the chart.",
        config=abl_cfg,
        task_type="Graph_Chart",
        max_mark=10.0,
        task_id="CHART_001",
        student_id="ablation_student",
        verbose=False
    )
    cap_str = r.cap_reason if r.cap_applied else "None"
    print(f"{desc:<45} | {r.total_score:<12.2f} | {r.performance_band:<10} | {cap_str:<18}")

print("="*90)

In [ ]:
# Step 7: Ground Truth Benchmarking against evaluation.csv
gt_csv = "evaluation.csv"
if os.path.exists(gt_csv) and os.path.exists(output_csv):
    metrics = benchmark_against_ground_truth(output_csv, gt_csv)
    print("\n" + "="*50)
    print("BENCHMARK AGAINST GROUND TRUTH (evaluation.csv)")
    print("="*50)
    for k, v in metrics.items():
        print(f"  - {k:<25}: {v}")
    print("="*50)